## base import and API modules

In [ ]:
import requests
import json
from faker import Faker

from contextlib import redirect_stdout
import io

# Global variables for pagination, easy to tweak here
DEFAULT_PAGE = 0
DEFAULT_SIZE = 25
BASE_URL = "http://localhost:8080/api"

general_user = "connercharles"
general_user_2 = "nicole09"
general_game = "Undertale"

def print_response(response):
    """Utility to print responses clearly (indented JSON or plain string)"""
    print(f"Status Code: {response.status_code}")
    try:
        # Try to parse and print nicely formatted JSON
        parsed_json = response.json()
        print(json.dumps(parsed_json, indent=4))
    except ValueError:
        # If it fails (e.g., plain text like "The game has been added"), print raw text
        text = response.text
        if text:
            print(text)
        else:
            print("[No content in response body]")
    print("=" * 50 + "\n")

def print_json(target_json):
    print(json.dumps(target_json, indent=4))

def mute(func, *args, **kwargs):
    """Prevents output of function func"""
    with redirect_stdout(io.StringIO()):
        return func(*args, **kwargs)

def login_as(username):
    api.logout()
    my_payload = {
        "email" : username + "@hotmail.com",
        "password" : username
    }
    response = api.login(my_payload)
    if(response.status_code != 200):
        raise ValueError("ERROR DURING LOGIN")
    else:
        print("You are now logged as: " + username)

def admin_login():
    api.logout()
    my_payload = {
        "email" : "admin@admin.com",
        "password" : "admin"
    }
    response = api.login(my_payload)
    if(response.status_code != 200):
        raise ValueError("ERROR DURING LOGIN")
    else:
        print("You are now logged as Admin")
    

def get_user_id(username):
    src_res = api.search_user(username,0,1).json()["content"]
    if(len(src_res) != 1):
        raise ValueError("USER NOT FOUND")
    return src_res[0]["id"]

def get_game_id(game_name):
    src_res = api.search_game(game_name,0,1).json()["content"]
    if(len(src_res) != 1):
        raise ValueError("GAME NOT FOUND")
    return src_res[0]["id"]

def get_user_game_review(user,game_id, can_fail = True):
    review_id = ""

    response = api.get_game_reviews(game_id).json()

    for r in response:
        if(r["username"] == user):
            review_id = r["id"]
            break

    if(review_id == "" and can_fail):
        raise ValueError("NO MATCHING REVIEW WAS FOUND")
    
    return review_id

def create_unique_user():
    fake = Faker()
    for i in range(0,5):
        while(True):
            register_username = fake.user_name() # avoid duplicates
            user_payload = {
                "username" : register_username,
                "email" : register_username + "@hotmail.com",
                "password" : register_username,
                "birthDate" : "2002-08-29"
            }

            response = api.register(user_payload)
            if(response.status_code == 200):
                break
        return register_username

class PlayerHiveClient:
    def __init__(self, base_url):
        self.base_url = base_url
        self.session = requests.Session()
        self.session.headers.update({"Content-Type": "application/json"})
        self.token = None

    def set_token(self, token):
        """Sets the JWT token for authenticated requests"""
        self.token = token
        self.session.headers.update({"Authorization": f"Bearer {token}"})
        print("[+] Token set successfully. You are now authenticated.\n")

    def logout(self):
        """Clears the token to allow switching accounts or testing unauthenticated routes"""
        self.token = None
        if "Authorization" in self.session.headers:
            del self.session.headers["Authorization"]
        print("[-] Token cleared. You are now logged out.\n")

    def _request(self, method, endpoint, **kwargs):
        """Base method to execute and print the request"""
        url = f"{self.base_url}{endpoint}"
        print(f"--- {method} {url} ---")
        response = self.session.request(method, url, **kwargs)
        return response

#### API MODULES
class API(PlayerHiveClient):
    
    # ================= AUTH =================
    def register(self, payload={}):
        return self._request("POST", "/auth/register", json=payload)

    def login(self, payload={}):
        response = self._request("POST", "/auth/login", json=payload)
        if response.status_code == 200:
            token = response.json().get("accessToken")
            if token:
                self.set_token(token)
        return response

    # ================= USER =================
    def get_own_profile(self):
        return self._request("GET", "/user/MyProfile")

    def get_user_profile(self, user_id):
        return self._request("GET", f"/user/{user_id}")
    
    def get_own_library(self,page=DEFAULT_PAGE, size=DEFAULT_SIZE):
        params = {"page": page, "size": size}
        return self._request("GET", f"/user/MyLibrary", params=params)

    def get_user_library(self, user_id, page=DEFAULT_PAGE, size=DEFAULT_SIZE):
        params = {"page": page, "size": size}
        return self._request("GET", f"/user/library/{user_id}", params=params)

    def edit_library(self, payload={}):
        return self._request("POST", "/user/editLibrary", json=payload)

    def remove_from_library(self, game_id):
        return self._request("DELETE", f"/user/removeFromLibrary/{game_id}")

    def get_friend_list(self, user_id, page=DEFAULT_PAGE, size=DEFAULT_SIZE):
        params = {"page": page, "size": size}
        return self._request("GET", f"/user/friends/{user_id}", params=params)
    
    def get_own_friend_list(self,page=DEFAULT_PAGE, size=DEFAULT_SIZE):
        params = {"page": page, "size": size}
        return self._request("GET", f"/user/MyFriends", params=params)
    

    def get_friend_requests(self,page=DEFAULT_PAGE,size=DEFAULT_SIZE):
        params = {"page": page, "size": size}
        return self._request("GET", "/user/friendRequests", params = params)

    def search_user(self, query, page=DEFAULT_PAGE, size=DEFAULT_SIZE):
        params = {"page": page, "size": size}
        return self._request("GET", f"/user/search/{query}", params=params)

    def send_friend_request(self, target_user_id):
        return self._request("POST", f"/user/sendFriendRequest/{target_user_id}")

    def approve_friend_request(self, target_user_id):
        return self._request("POST", f"/user/approveFriendRequest/{target_user_id}")

    def deny_friend_request(self, target_user_id):
        return self._request("DELETE", f"/user/denyFriendRequest/{target_user_id}")

    def remove_friend(self, friend_id):
        return self._request("DELETE", f"/user/removeFriend/{friend_id}")
    
    def get_user_reviews(self, user_id, page=DEFAULT_PAGE,size=DEFAULT_SIZE):
        params = {"page": page, "size": size}
        return self._request("GET","/user/reviews/" + user_id, params = params)
    
    def get_own_reviews(self, page=DEFAULT_PAGE,size=DEFAULT_SIZE):
        params = {"page": page, "size": size}
        return self._request("GET","/user/MyReviews", params = params)

    def delete_account(self):
        return self._request("DELETE", "/user/deleteAccount")

    # ================= GAMES =================
    def get_game_info(self, game_id):
        return self._request("GET", f"/games/{game_id}")

    def search_game(self, game_name, page=DEFAULT_PAGE, size=DEFAULT_SIZE):
        params = {"page": page, "size": size}
        return self._request("GET", f"/games/search/{game_name}", params=params)

    def get_game_reviews(self, game_id, page=DEFAULT_PAGE, size=DEFAULT_SIZE):
        params = {"page": page, "size": size}
        return self._request("GET", f"/games/showReviews/{game_id}", params=params)

    def add_review(self, game_id, payload={}):
        return self._request("POST", f"/games/addReview/{game_id}", json=payload)

    def delete_review(self, review_id):
        return self._request("DELETE", f"/games/deleteReview/{review_id}")

    # ================= ADMIN =================
    def add_game(self, payload={}):
        return self._request("POST", "/admin/addGame", json=payload)

    def edit_game(self, game_id, payload={}):
        return self._request("PATCH", f"/admin/editGame/{game_id}", json=payload)

    def delete_game(self, game_id):
        return self._request("DELETE", f"/admin/deleteGame/{game_id}")

    def delete_user_admin(self, user_id):
        return self._request("DELETE", f"/admin/deleteUser/{user_id}")

# Initialize the API object
api = API(BASE_URL)


## LOGIN SECTION

### Register user

In [ ]:
print(">>> REGISTER AS STANDARD USER")
fake = Faker()

register_username = fake.user_name() # avoid duplicates
user_payload = {
    "username" : register_username,
    "email" : register_username + "@hotmail.com",
    "password" : register_username,
    "birthDate" : "2002-08-29"
}
print(">>>>> Registering as: "+register_username)
print_response(api.register(user_payload))

print(">>> SAME USERNAME TEST")

user_payload = {
    "username" : register_username,
    "email" : "eyes_emoji" + "@hotmail.com",
    "password" : register_username,
    "birthDate" : "2002-08-29"
}

print_response(api.register(user_payload))

print(">>> SAME EMAIL TEST")

user_payload = {
    "username" : "eyes_emoji",
    "email" : register_username + "@hotmail.com",
    "password" : register_username,
    "birthDate" : "2002-08-29"
}

print_response(api.register(user_payload))

print(">>> FRESH USER DELETION")
login_as(register_username)
print_response(api.delete_account())


### General Login

In [ ]:
api.logout()

print(">>> LOGIN AS STANDARD USER")
username = "fabriziomaggi"
user_payload = {
    "email" : username + "@hotmail.com",
    "password" : username

}
print_response(api.login(user_payload))

## USER CONTROLLER TESTS

### Profile API tests

In [ ]:
print(">>> USER SEARCH")

api.logout()

page_number = 0
page_size = 10

user_search_target_string = "br"
user_search_result = api.search_user(user_search_target_string,page_number,page_size)

print_response(user_search_result)
# print_json(user_search_result.json()["content"])

print(">>> EMPTY USER SEARCH")

user_search_target_string = "&&&&&"
user_search_result = api.search_user(user_search_target_string,page_number,page_size)

print_response(user_search_result)

In [ ]:
print(">>> PAGINATION VALUES TEST")

print(">>> NEGATIVE PAGE NUMBER")

api.logout()

page_number = -1
page_size = 10

user_search_target_string = "br"
user_search_result = api.search_user(user_search_target_string,page_number,page_size)

print_response(user_search_result)

print(">>> PAGE SIZE 0")

page_number = 0
page_size = 0

user_search_result = api.search_user(user_search_target_string,page_number,page_size)

print_response(user_search_result)

print(">>> NEGATIVE PAGE SIZE")

page_number = 0
page_size = -1

user_search_result = api.search_user(user_search_target_string,page_number,page_size)

print_response(user_search_result)

print(">>> OUT OF BOUNDS PAGE NUMBER VALUE")

page_number = 100000
page_size = 10

user_search_result = api.search_user(user_search_target_string,page_number,page_size)

print_response(user_search_result)

print(">>> OUT OF BOUNDS PAGE SIZE VALUE")

page_number = 0
page_size = 10000

user_search_result = api.search_user(user_search_target_string,page_number,page_size)

print_response(user_search_result)

In [ ]:
print(">>> PROFILE REQUEST")

api.logout()

#username = "wgarcia"
username = general_user

user_id = get_user_id(username)

print_response(api.get_user_profile(user_id))

In [ ]:
print(">>> NON EXISTENT PROFILE REQUEST")

api.logout()

print_response(api.get_user_profile("tomato"))

In [ ]:
print(">>> OWN PROFILE REQUEST, BOTH WITH ID AND WITHOUT")

#username = ""
username = general_user

login_as(username)

user_id = get_user_id(username)

my_profile_without_id = api.get_user_profile(user_id)
my_profile_with_id = api.get_own_profile()

print_response(my_profile_with_id)
if(my_profile_without_id.json() == my_profile_with_id.json()):
    print("=========== The two queries are equal ==========")
else:
    print("======= ERROR! WITHOUT ID:")
    print_response(my_profile_without_id)

In [ ]:
print(">>> \"MY\" QUERIES WITH NO ACCOUNT")

api.logout()

print_response(api.get_own_profile())
print_response(api.get_own_library(0,10))
print_response(api.get_own_reviews(0,10))

In [ ]:
print(">>> LIBRARY REQUEST")

api.logout()

#username = ""
username = general_user

target_user = get_user_id(username)

page_number = 0
page_size = 11

print_response(api.get_user_library(target_user, page_number, page_size))

In [ ]:
print(">>> OWN LIBRARY REQUEST")

#username = ""
username = general_user

login_as(username)

page_number = 0
page_size = 2

print_response(api.get_own_library( page_number, page_size))

In [ ]:
print(">>> NON EXISTENT LIBRARY REQUEST")

api.logout()

page_number = 0
page_size = 2

print_response(api.get_user_library("tomato", page_number, page_size))

In [ ]:
print(">>> ADDING GAME TO LIBRARY")

username = general_user
#username = ""

game_name = general_game
#game_name = "Half-Life"
achievements = 0
hours = 10
hours2 = 100

# ADD QUERY PRINT

login_as(username)

game_id = get_game_id(game_name)

payload = {
    "gameId" : game_id,
    "achievements" : achievements,
    "hoursPlayed" : hours
}

print_response(api.edit_library(payload))


print(">>> EDIT GAME IN LIBRARY")

payload["hoursPlayed"] = hours2

print_response(api.edit_library(payload))

print(">>> REMOVE GAME FROM LIBRARY")

print_response(api.remove_from_library(game_id))


In [ ]:
print(">>> NO ACCOUNT LIBRARY MANIPULATION")

api.logout()

payload ={
    "gameId":"tomato",
    "achievements":"0",
    "hoursPlayed":"10"
}

print_response(api.edit_library(payload))


print(">>> GHOST GAME LIBRARY MANIPULATION")

login_as(general_user)

print_response(api.edit_library(payload))


print(">>> GHOST GAME REMOVAL")

print_response(api.remove_from_library("tomato"))


print(">>> BAD ACHIEVEMENT VALUE")

payload["gameId"] = get_game_id(general_game)
payload["achievements"] = 9999
print_response(api.edit_library(payload))

payload["achievements"] = -1
print_response(api.edit_library(payload))




In [ ]:
print(">>> PAGED FRIEND LIST")

username = general_user

page_number = 0
page_size = 3

api.logout()

user_id = get_user_id(username)

print_response(api.get_friend_list(user_id,page_number,page_size))

print(">>> OWN PAGED FRIEND LIST")

login_as(general_user)

print_response(api.get_own_friend_list(page_number,page_size))

print(">>> NO LOGIN MY FRIEND LIST")

api.logout()

print_response(api.get_own_friend_list(page_number,page_size))

print(">>> NO USER FRIEND LIST")

print_response(api.get_friend_list("thehatman",page_number,page_size))


In [ ]:
print(">>> PAGED FRIEND REQUESTS")

#username = ""
username = general_user

login_as(username)

page_number = 0
page_size = 2

print_response(api.get_friend_requests(page_number,page_size))

print(">>> NO LOGIN FRIEND REQUESTS")

api.logout()

print_response(api.get_friend_requests(page_number,page_size))


In [ ]:
print(">>> REVERSE ARRAY PAGINATION TEST")

username = general_user

login_as(username)

print(">>> NEGATIVE PAGE NUMBER")

page_number = -1
page_size = 2

print_response(api.get_friend_requests(page_number,page_size))

print(">>> SIZE OUT OF BOUNDS")

page_number = 0
page_size = 10000

print_response(api.get_friend_requests(page_number,page_size))

print(">>> SIZE ZERO")

page_number = 0
page_size = 0

print_response(api.get_friend_requests(page_number,page_size))

print(">>> NEGATIVE SIZE")

page_number = 0
page_size = -1

print_response(api.get_friend_requests(page_number,page_size))

print(">>> OUT OF BOUNDS TEST")

page_number = 100
page_size = 10

print_response(api.get_friend_requests(page_number,page_size))


In [ ]:
print(">>> SEVERAL FRIEND MANAGEMENT TESTS")

user1 = general_user
user2 = general_user_2

api.logout()

user1_id = get_user_id(user1)
user2_id = get_user_id(user2)

if(user1_id == user2_id):
    raise ValueError("THE USERS ARE THE SAME!")

print(">>>> NEO4J QUERY: MATCH (n:User {username:\""+user1+"\"}"+
      ")-[r:FRIENDS_WITH]-(g:User) MATCH (n2:User"+
      " {username:\""+user2+"\"})-[r2:FRIENDS_WITH]-(g2:User) "+
      "RETURN n,r,g, n2, r2, g2;")

# clear all users status
print(">>> FRIENDSHIP CLEANUP") # yes i know it is dumb to use the functions that we are going to test
login_as(user1)
api.remove_friend(user2_id)
api.deny_friend_request(user2_id)

login_as(user2)
api.deny_friend_request(user1_id)

print()
print(">>> SEND FRIEND REQUEST")

login_as(user1)

print_response(api.send_friend_request(user2_id))

print(">>> DENY FRIEND REQUEST")

login_as(user2)

print_response(api.deny_friend_request(user1_id))

print(">>> ACCEPT FRIEND REQUEST")

login_as(user1)

api.send_friend_request(user2_id)

login_as(user2)

print_response(api.approve_friend_request(user1_id))

print(">>> REMOVE FRIEND")

login_as(user1)

print_response(api.remove_friend(user2_id))

print(">>> TWO-WAY FRIEND REQUEST")

login_as(user1)

print_response(api.send_friend_request(user2_id))

login_as(user2)

print_response(api.send_friend_request(user1_id))

print_response(api.remove_friend(user1_id))

print(">>> DOUBLE FRIEND REQUEST")

print_response(api.send_friend_request(user1_id))
print_response(api.send_friend_request(user1_id))

login_as(user1)
api.deny_friend_request(user2_id)


In [ ]:
print(">>> NO LOGIN FRIEND MANAGEMENT")

api.logout()

user_id = get_user_id(general_user)
ghost_id = "brotato"

print(">>>>> SEND")
print_response(api.send_friend_request(user_id))

print(">>>>> ACCEPT")
print_response(api.approve_friend_request(user_id))

print(">>>>> DENY")
print_response(api.deny_friend_request(user_id))

print(">>>>> REMOVE FRIEND")
print_response(api.remove_friend(user_id))

print(">>> SELF FRIEND REMOVAL")
login_as(general_user)

print_response(api.remove_friend(user_id))

print(">>> SELF FRIEND REQUEST")

print_response(api.send_friend_request(user_id))

print(">>> SEND FRIEND REQUEST TO NO ONE")

print_response(api.send_friend_request(ghost_id))

print(">>> ACCEPT GHOST FRIEND REQUEST")
print_response(api.approve_friend_request(ghost_id))




In [ ]:
print(">>> PAGED USER REVIEWS")

api.logout()

#username = ""
username = general_user
user_id = get_user_id(username)

page_number =0
page_size = 8

print_response(api.get_user_reviews(user_id,page_number,page_size))

print(">>> OWN REVIEWS")

login_as(username)

print_response(api.get_own_reviews(page_number,page_size))

MISSING: TEST OWN ACCOUNT DELETE

## GAME CONTROLLER TEST

In [ ]:
print(">>> GAME REQUEST BY ID")

api.logout()

game_id = get_game_id(general_game)

response1 = api.get_game_info(game_id)

print_response(response1)

print(">>> LOGIN GAME TEST")

login_as(general_user)

response2 = api.get_game_info(game_id)

if(response1.json() != response2.json()):
    raise ValueError("WRONG RESPONSE")
else:
    print()
    print(">>>>>>> CORRECT RESPONSE")
    print()

In [ ]:
api.logout()

print(">>> GHOST GAME TEST")

print_response(api.get_game_info("tomato"))

In [ ]:
api.logout()

print(">>> SEARCH GAME TEST")

game_target_string = "under"
page_number = 0
page_size = 3

print_response(api.search_game(game_target_string, page_number, page_size))

print(">>> EMPTY GAME SEARCH")

print_response(api.search_game("&&&&", page_number, page_size))


In [ ]:
api.logout()

print(">>> GAME REVIEWS REQUEST TEST")

game = general_game
game_id = get_game_id(general_game)

page_number = 0
page_size = 25

print_response(api.get_game_reviews(game_id,page_number,page_size))

In [ ]:
api.logout()

game = general_game
user = general_user

game_id = get_game_id(game)
login_as(user)

review = {
    "reviewText" : "i like apples, tangerines and "+ game,
    "score" : 9.5
}

# cleanup

review_id = get_user_game_review(user,game_id,False)

if(review_id != ""):
    api.delete_review(review_id)


print(">>> ADD GAME REVIEW TEST")

print_response(api.add_review(game_id,review))

print(">>> DOUBLE REVIEW TEST")

print_response(api.add_review(game_id,review))

print(">>> DELETE GAME REVIEW TEST")

# this function can't seem to find the review
review_id = get_user_game_review(user,game_id)

print_response(api.delete_review(review_id))

In [ ]:
login_as(general_user)

impostor = general_user_2

review = {
    "reviewText" : "i like apples, tangerines and "+ game,
    "score" : 9.5
}

print_response(api.add_review(get_game_id(general_game),review))

review_id = get_user_game_review(user,game_id,False)

api.logout()

print(">>> NO LOGIN REVIEWS MANIPULATION")

print_response(api.add_review(game_id,review))

print_response(api.delete_review(review_id))

print(">>> WRONG USER ATTEMPTING DELETE")
login_as(impostor)
print_response(api.delete_review(review_id))


# cleanup
login_as(general_user) 
api.delete_review(review_id)

## ADMIN CONTROLLER

In [ ]:
admin_login()

fake = Faker()
name = fake.company()
print("Adding game: " + name)
payload = {
    "name": name,
    "releaseDate": "2002-08-29",
    "price": 100,
    "discount": 30,
    "description": "fortnite is bad",
    "imageURL": "obama.com",
    "supportedOS": [
        "Windows",
        "Linux"
    ],
    "achievements": 100,
    "developers": [
        "obamna"
    ],
    "publishers": [
        "epic games",
        "israel"
    ],
    "genres": [
        "super obama"
    ]
}

edit_payload = {
    "discount":35,
    "description":"fortnite is good",
    "imageURL":"obamadotcom",
    "supportedOS":["Windows","Linux","mac"],
    "achievements":300,
    "developers":["netanyahu"],
    "publishers":["israel"]
}

print(">>> ADD NEW GAME")
print_response(api.add_game(payload))

game_id = get_game_id(name)


print(">>> REQUESTING GAME INFO")

print_response(api.get_game_info(game_id))


print(">>> ADDING EXISTING GAME")
print_response(api.add_game(payload))

print(">>> EDIT GAME")
print_response(api.edit_game(game_id, edit_payload))


print(">>> EDIT GAME TO SAME NAME")

edit_payload["name"] = name
print_response(api.edit_game(game_id, edit_payload))


print(">>> EDIT GAME TO EXISTING NAME")

edit_payload["name"] = general_game
print_response(api.edit_game(game_id, edit_payload))

print(">>> FRESH GAME DELETE")
print_response(api.delete_game(game_id))

print(">>> DELETE NON-EXISTING GAME")
print_response(api.delete_game(game_id))


In [ ]:
api.logout()

print(">>> DELETE USER AS ADMIN")

register_username = fake.user_name() # avoid duplicates
user_payload = {
    "username" : register_username,
    "email" : register_username + "@hotmail.com",
    "password" : register_username,
    "birthDate" : "2002-08-29"
}

print(">>>> NEW USER IS: " + register_username)
api.register(user_payload)

user_id = get_user_id(register_username)

admin_login()

print(">>> DELETE FRESH USER")
print_response(api.delete_user_admin(user_id))

print(">>> DELETE NON-EXISTENT USER")
print_response(api.delete_user_admin(user_id))


print(">>> REVIEW DELETION TEST")

# we first add a review to delete
game = general_game
user = general_user

game_id = get_game_id(game)
login_as(user)

review = {
    "reviewText" : "i like apples, tangerines and "+ game,
    "score" : 9.5
}

api.add_review(game_id,review)
review_id = get_user_game_review(user,game_id,False)

admin_login()
print_response(api.delete_review(review_id))



In [ ]:
print(">>> ADMIN AUTH TEST")
login_as(general_user)

user_id = get_user_id(general_user_2)
game_id = get_game_id(general_game)

print_response(api.add_game({"payload" : "fake"}))
print_response(api.edit_game({"payload" : "fake"}))
print_response(api.delete_game(game_id))
print_response(api.delete_user_admin(user_id))

In [ ]:
print(">>> HEAVILY CONNECTED GAME DELETION")

connected_users = 5

admin_login()

fake = Faker()
name = fake.company()
print("Adding game: " + name)
payload = {
    "name": name,
    "releaseDate": "2002-08-29",
    "price": 100,
    "discount": 30,
    "description": "fortnite is bad",
    "imageURL": "obama.com",
    "supportedOS": [
        "Windows",
        "Linux"
    ],
    "achievements": 100,
    "developers": [
        "obamna"
    ],
    "publishers": [
        "epic games",
        "israel"
    ],
    "genres": [
        "super obama"
    ]
}

review = {
    "reviewText" : "i like apples, tangerines and "+ name,
    "score" : 9.5
}

api.add_game(payload)
game_id = get_game_id(name)

library = {
    "gameId" : game_id,
    "achievements" : 100,
    "hoursPlayed" : 100
}

api.logout()

query_string = "{ username: { $in: [" 

for i in range(0,connected_users):
    
    u = create_unique_user()

    login_as(u)
    api.add_review(game_id,review)
    api.edit_library(library)
    query_string += "\"" + u + "\""
    if(i != connected_users - 1):
        query_string += ","

query_string += "]} }"

print()
print()
print("MONGODB QUERY:")
print(query_string)
print()
print()
admin_login()

print_response(api.delete_game(game_id))

In [ ]:
api.logout()

friends = 5


anchor = create_unique_user()
anchor_id = get_user_id(anchor)

print("Anchor is: " + anchor)

query_string = "{ username: { $in: [" 

query_string += "\"" + anchor + "\""
if(friends > 0):
    query_string += ","

user_ids = []

# sending all friend requests
for i in range(0,friends):
    u = create_unique_user()
    
    login_as(u)

    api.send_friend_request(anchor_id)
    user_ids.append(get_user_id(u))

    query_string += "\"" + u + "\""
    if(i != friends - 1):
        query_string += ","

# accepting friend requests
login_as(anchor)

for i in range(0, friends):
    api.approve_friend_request(user_ids[i])

if(api.get_own_profile().json()["friends"] != friends):
    raise ValueError("WRONG FRIEND VALUE")
else:
    print()
    print()
    print(f">>>>> {friends} FRIENDSHIPS ESTABLISHED!")


query_string += "]} }"

print()
print()
print("MONGODB QUERY:")
print(query_string)
print()
print()

print("NEO4J QUERY:")
print("MATCH (u1:User { username : \"" + anchor + "\" })-[r:FRIENDS_WITH]-"
      +"(u2:User) RETURN u1, u2, r;")
print()
print()

admin_login()

print(">>> DELETING ANCHOR")

print_response(api.delete_user_admin(anchor_id))
